<a href="https://colab.research.google.com/github/AlexDorosh55/Practical_DL/blob/fall25/week07_attention/homework_transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Homework 5: Question search engine

Remeber week01 where you used GloVe embeddings to find related questions? That was.. cute, but far from state of the art. It's time to really solve this task using context-aware embeddings.

__Warning:__ this task assumes you have seen `seminar.ipynb`!

In [28]:
%pip install --upgrade transformers datasets accelerate deepspeed
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
import datasets
from tqdm import tqdm
import os
import tempfile
import pandas as pd
import time
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

### Load data and model

In [2]:
qqp = datasets.load_dataset('SetFit/qqp')
print('\n')
print("Sample[0]:", qqp['train'][0])
print("Sample[3]:", qqp['train'][3])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/313 [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


train.jsonl:   0%|          | 0.00/70.8M [00:00<?, ?B/s]

validation.jsonl: 0.00B [00:00, ?B/s]

test.jsonl:   0%|          | 0.00/76.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/363846 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/40430 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/390965 [00:00<?, ? examples/s]



Sample[0]: {'text1': 'How is the life of a math student? Could you describe your own experiences?', 'text2': 'Which level of prepration is enough for the exam jlpt5?', 'label': 0, 'idx': 0, 'label_text': 'not duplicate'}
Sample[3]: {'text1': 'What can one do after MBBS?', 'text2': 'What do i do after my MBBS ?', 'label': 1, 'idx': 3, 'label_text': 'duplicate'}


In [3]:
model_name = "gchhablani/bert-base-cased-finetuned-qqp"
tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
model = transformers.AutoModelForSequenceClassification.from_pretrained(model_name)

tokenizer_config.json:   0%|          | 0.00/320 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/433M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

### Tokenize the data

In [4]:
MAX_LENGTH = 128
def preprocess_function(examples):
    result = tokenizer(
        examples['text1'], examples['text2'],
        padding='max_length', max_length=MAX_LENGTH, truncation=True
    )
    result['label'] = examples['label']
    return result

qqp_preprocessed = qqp.map(preprocess_function, batched=True)

Map:   0%|          | 0/363846 [00:00<?, ? examples/s]

Map:   0%|          | 0/40430 [00:00<?, ? examples/s]

Map:   0%|          | 0/390965 [00:00<?, ? examples/s]

In [5]:
print(repr(qqp_preprocessed['train'][0]['input_ids'])[:100], "...")

[101, 1731, 1110, 1103, 1297, 1104, 170, 12523, 2377, 136, 7426, 1128, 5594, 1240, 1319, 5758, 136,  ...


### Task 1: evaluation (1 point)

We randomly chose a model trained on QQP - but is it any good?

One way to measure this is with validation accuracy - which is what you will implement next.

Here's the interface to help you do that:

In [6]:
val_set = qqp_preprocessed['validation']
val_loader = torch.utils.data.DataLoader(
    val_set, batch_size=1, shuffle=False, collate_fn=transformers.default_data_collator
)

In [7]:
for batch in val_loader:
     break  # here be your training code
print("Sample batch:", batch)

with torch.no_grad():
    predicted = model(
        input_ids=batch['input_ids'],
        attention_mask=batch['attention_mask'],
        token_type_ids=batch['token_type_ids']
    )

print('\nPrediction (probs):', torch.softmax(predicted.logits, dim=1).data.numpy())

Sample batch: {'labels': tensor([0]), 'idx': tensor([0]), 'input_ids': tensor([[  101,  2009,  1132,  2170,   118,  4038,  1177,  2712,   136,   102,
          2009,  1132,  1117, 10224,  4724,  1177,  2712,   136,   102,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,   

__Your task__ is to measure the validation accuracy of your model.
Doing so naively may take several hours. Please make sure you use the following optimizations:

- run the model on GPU with no_grad
- using batch size larger than 1
- use optimize data loader with num_workers > 1
- (optional) use [mixed precision](https://pytorch.org/docs/stable/notes/amp_examples.html)


In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.eval()
model.to(device);

In [9]:
val_set = qqp_preprocessed['validation']
val_loader = torch.utils.data.DataLoader(
    val_set,
    batch_size=64,
    shuffle=False,
    collate_fn=transformers.default_data_collator,
    num_workers=4,
    pin_memory=True
)

total_correct = 0
total_samples = 0

with torch.no_grad():
    for batch in tqdm(val_loader, desc="Validating"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        token_type_ids = batch['token_type_ids'].to(device)
        labels = batch['labels'].to(device)

        with torch.amp.autocast(device_type=device.type, dtype=torch.float16, enabled=(device.type  == 'cuda')):
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids
            )
            logits = outputs.logits
        predictions = torch.argmax(logits, dim=1)
        total_correct += (predictions == labels).sum().item()
        total_samples += labels.size(0)

accuracy = total_correct / total_samples

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
Validating: 100%|██████████| 632/632 [01:01<00:00, 10.23it/s]


In [10]:
assert 0.9 < accuracy < 0.91

### Task 2: train the model (4 points)

For this task, you have two options:

__Option A:__ fine-tune your own model. You are free to choose any model __except for the original BERT.__ We recommend [DeBERTa-v3](https://huggingface.co/microsoft/deberta-v3-base). Better yet, choose the best model based on public benchmarks (e.g. [GLUE](https://gluebenchmark.com/)).

You can write the training code manually or use transformers.Trainer (see [this example](https://github.com/huggingface/transformers/blob/main/examples/pytorch/text-classification)). Please make sure that your model's accuracy is at least __comparable__ with the above example for BERT.


__Option B:__ compare at least 3 pre-finetuned models (in addition to the above BERT model). For each model, report (1) its accuracy, (2) its speed, measured in samples per second in your hardware setup and (3) its size in megabytes. Please take care to compare models in equal setting, e.g. same CPU / GPU. Compile your results into a table and write a short (~half-page on top of a table) report, summarizing your findings.

In [11]:
model_names = [
    "gchhablani/bert-base-cased-finetuned-qqp",
    "0xb1/distilbert-base-uncased-finetuned-qqp",
    "JeremiahZ/roberta-base-qqp",
    "rambodazimi/deberta-v3-base-finetuned-FFT-QQP",
    "M-FAC/bert-tiny-finetuned-qqp"
]

In [12]:
qqp = qqp.rename_column("label", "labels")
val_set_original = qqp['validation']

In [18]:
results = []
for model_name in model_names:
    print(f"\nProcessing model: {model_name} ---")

    try:
        tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
        model = transformers.AutoModelForSequenceClassification.from_pretrained(model_name)
        model.to(device)
        model.eval()
        size_mb = 0
        with tempfile.TemporaryDirectory() as temp_dir:
            model.save_pretrained(temp_dir)
            for f in os.listdir(temp_dir):
                if f.endswith('.bin') or f.endswith('.safetensors'):
                    size_mb += os.path.getsize(os.path.join(temp_dir, f))
        size_mb = size_mb / (1024 * 1024)

        def preprocess_function(examples):
            return tokenizer(examples['text1'], examples['text2'], truncation=True, padding='max_length', max_length=128)

        val_set_processed = val_set_original.map(
            preprocess_function,
            batched=True
        )
        columns_to_set = ['input_ids', 'attention_mask', 'labels']
        if 'token_type_ids' in val_set_processed.features:
            columns_to_set.append('token_type_ids')

        cols_to_remove = [col for col in val_set_processed.column_names if col not in columns_to_set]
        val_set_processed = val_set_processed.remove_columns(cols_to_remove)
        val_set_processed.set_format(type='torch', columns=columns_to_set)
        val_loader = torch.utils.data.DataLoader(
            val_set_processed,
            batch_size=64,
            shuffle=False,
            collate_fn=transformers.default_data_collator,
            num_workers=4,
            pin_memory=True
        )
        total_correct = 0
        total_samples = 0

        start_time = time.time()

        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"Validating {model_name.split('/')[-1]}"):

                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['labels'].to(device)

                model_inputs = {
                    'input_ids': input_ids,
                    'attention_mask': attention_mask
                }

                if 'token_type_ids' in batch:
                    model_inputs['token_type_ids'] = batch['token_type_ids'].to(device)

                with torch.amp.autocast(device_type=device.type, dtype=torch.float16, enabled=(device.type == 'cuda')):
                    outputs = model(**model_inputs)
                    logits = outputs.logits

                predictions = torch.argmax(logits, dim=1)
                total_correct += (predictions == labels).sum().item()
                total_samples += labels.size(0)

        end_time = time.time()
        accuracy = (total_correct / total_samples) * 100
        duration = end_time - start_time
        speed_sps = total_samples / duration

        results.append({
            "Model": model_name,
            "Accuracy (%)": accuracy,
            "Speed (samples/sec)": speed_sps,
            "Size (MB)": size_mb
        })

    except Exception as e:
        print(f"!!! FAILED {model_name}: {e}")
        results.append({
            "Model": model_name,
            "Accuracy (%)": "Error",
            "Speed (samples/sec)": "Error",
            "Size (MB)": "Error"
        })


Processing model: gchhablani/bert-base-cased-finetuned-qqp ---


Map:   0%|          | 0/40430 [00:00<?, ? examples/s]

Validating bert-base-cased-finetuned-qqp: 100%|██████████| 632/632 [01:10<00:00,  9.02it/s]



Processing model: 0xb1/distilbert-base-uncased-finetuned-qqp ---


Map:   0%|          | 0/40430 [00:00<?, ? examples/s]

...

config.json:   0%|          | 0.00/759 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/17.6M [00:00<?, ?B/s]

Map:   0%|          | 0/40430 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/17.6M [00:00<?, ?B/s]

Validating bert-tiny-finetuned-qqp: 100%|██████████| 632/632 [00:05<00:00, 106.35it/s]


In [19]:
df = pd.DataFrame(results)
df['Accuracy (%)'] = pd.to_numeric(df['Accuracy (%)'], errors='coerce').map('{:,.2f}'.format)
df['Speed (samples/sec)'] = pd.to_numeric(df['Speed (samples/sec)'], errors='coerce').map('{:,.2f}'.format)
df['Size (MB)'] = pd.to_numeric(df['Size (MB)'], errors='coerce').map('{:,.2f}'.format)
print(df.to_markdown(index=False))

| Model                                         |   Accuracy (%) | Speed (samples/sec)   |   Size (MB) |
|:----------------------------------------------|---------------:|:----------------------|------------:|
| gchhablani/bert-base-cased-finetuned-qqp      |          90.84 | 576.74                |      413.2  |
| 0xb1/distilbert-base-uncased-finetuned-qqp    |          85.12 | 1,090.92              |      255.43 |
| JeremiahZ/roberta-base-qqp                    |          91.53 | 623.51                |      475.51 |
| rambodazimi/deberta-v3-base-finetuned-FFT-QQP |          92.55 | 352.01                |      703.54 |
| M-FAC/bert-tiny-finetuned-qqp                 |          84.4  | 6,800.16              |       16.74 |


#### Analysis

The clear leader is the **M-FAC/bert-tiny-finetuned-qqp** model prioritizing speed and size (memory). It demonstrates exceptional performance, processing **6,800 samples per second**, which is approximately **11.8 times faster** than the baseline BERT model (gchhablani/bert-base-cased...). Furthermore, its size is only **16.74 MB**, making it **24.7 times smaller** than the baseline (413.2 MB). However, this gain comes at the cost of accuracy: at **84.4%**, it has the lowest score in the group, trailing the baseline by 6.4%.

---

#### Balanced Alternative

As a balanced alternative, **0xb1/distilbert-base-uncased-finetuned-qqp (DistilBERT)** can be considered. This model is almost **2 times faster** than the baseline (1,090 samples/sec) and nearly **40% lighter** (255.43 MB). However, its accuracy (85.12%) only slightly surpasses BERT-tiny and is still far from the leaders.

---

#### High-Accuracy Models

Models with high accuracy, such as **JeremiahZ/roberta-base-qqp (91.53%)** and **rambodazimi/deberta-v3-base-finetuned-FFT-QQP (92.55%)**, significantly lag in performance. DeBERTa, while being the most accurate, is also the slowest (352 samples/sec) and the largest (703.54 MB).

### Task 3: try the full pipeline (1 point)

Finally, it is time to use your model to find duplicate questions.
Please implement a function that takes a question and finds top-5 potential duplicates in the training set. For now, it is fine if your function is slow, as long as it yields correct results.

Showcase how your function works with at least 5 examples.

In [21]:
model_name = "M-FAC/bert-tiny-finetuned-qqp"
tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
model = transformers.AutoModelForSequenceClassification.from_pretrained(model_name)
model.to(device)
model.eval();

In [22]:
all_text1 = qqp['train']['text1']
all_text2 = qqp['train']['text2']
unique_questions_set = set(all_text1)
unique_questions_set.update(all_text2)
search_space = list(unique_questions_set)

In [23]:
def find_top_k_duplicates(query_question, model, tokenizer, search_space, device, top_k=5, batch_size=64):

    results = []

    with torch.no_grad():
        for i in tqdm(range(0, len(search_space), batch_size), desc="Поиск"):

            batch_questions = search_space[i : i + batch_size]
            batch_pairs = [(query_question, q) for q in batch_questions]

            inputs = tokenizer(
                batch_pairs,
                padding=True,
                truncation=True,
                max_length=128,
                return_tensors="pt"
            )

            inputs = {k: v.to(device) for k, v in inputs.items()}

            with torch.amp.autocast(device_type=device.type, dtype=torch.float16, enabled=(device.type == 'cuda')):
                outputs = model(**inputs)
                logits = outputs.logits

            duplicate_scores = logits[:, 1]

            for q, score in zip(batch_questions, duplicate_scores.cpu().numpy()):
                results.append((score, q))

    results.sort(key=lambda x: x[0], reverse=True)

    return results[:top_k]

In [26]:
example_queries = [
        "What is the step by step guide to invest in markets in Russia?",
        "Why is the sky blue?",
        "What is the physical mechanism behind sonoluminescence?",
        "What is the meaning of life?",
        "How does the philosophical concept of 'qualia' challenge physicalist theories of mind?",
]

all_showcase_results = []
for query in example_queries:
    top_5_results = find_top_k_duplicates(
        query,
        model,
        tokenizer,
        search_space,
        device,
        top_k=5,
        batch_size=128
    )

    all_showcase_results.append({
        "Query": query,
        "Results": top_5_results
    })


Поиск: 100%|██████████| 3859/3859 [01:50<00:00, 35.07it/s]


In [27]:
for item in all_showcase_results:
    print(f"Query: {item['Query']}")
    df = pd.DataFrame(item['Results'], columns=['Logit', 'Found question'])
    df['Logit'] = df['Logit'].round(2)
    print(df.to_string(index=False))

Query: What is the step by step guide to invest in markets in Russia?
    Logit                                                            Found question
-0.270020                                How will Hillary Clinton deal with russia?
-0.300049                     Will Hillary Clinton start a nuclear war with Russia?
-0.320068              Would Hillary Clinton start World War III / War with Russia?
-0.350098         Why do you think Russia is trying to help Trump win the election?
-0.379883 Will Hillary Clinton make decisions that escalate into a war with Russia?
Query: Why is the sky blue?
   Logit                                                                                                 Found question
1.099609                                                                                Why is that the sky is so blue?
0.990234 Why does the sky appear blue? Why is the colour of the sky at the horizon different during sunrise and sunset?
0.959961                              

/usr/local/lib/python3.12/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.12/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.12/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.12/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.12/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


Okay, here are brief comments on each case.:

1. **Query (Invest in Russia):** The model ignored the key topic of "investment" and mistakenly "stuck" to the word "Russia" in a completely different, political context (Clinton, Trump). Negative logits (low confidence) rightly show that these are bad matches.

2.  **Query (Why is the sky blue?):** The model has successfully found exact semantic duplicates (paraphrases) of the question. Very high positive logits (0.9–1.1) show maximum confidence in similarity.

3. **Query (Sonoluminescence):** The request is too highly specialized. The model probably did not encounter "sonoluminescence" in the training data. She couldn't find anything similar and asked random questions about "death" and "conspiracies." Very low logits (<-0.7) confirm this.

4. **Query (Meaning of life):** The model correctly found semantically similar questions about the "purpose" of life. **However** one completely irrelevant question got into the output  with the same high logit (1.79).

5.  **Query (Qualia):** As in the case of sonoluminescence, the request is too academic. The model did not understand the terms "qualia" or "physicalist theories". Instead, she seems to have latched onto the word "theories" and mistakenly linked it to "conspiracy theories." Low logits (<-0.4) again show that the model is not sure of the result.

__Bonus:__ for bonus points, try to find a way to run the function faster than just passing over all questions in a loop. For isntance, you can form a short-list of potential candidates using a cheaper method, and then run your tranformer on that short list. If you opted for this solution, please keep both the original implementation and the optimized one - and explain briefly what is the difference there.

In [29]:
tfidf_vectorizer = TfidfVectorizer(lowercase=True)
search_space_tfidf = tfidf_vectorizer.fit_transform(search_space)

In [30]:
def find_top_k_duplicates_optimized(
    query_question,
    model,
    tokenizer,
    search_space,
    device,
    tfidf_vectorizer,
    search_space_tfidf,
    top_k=5,
    batch_size=64,
    candidate_list_size=100
):
    query_tfidf = tfidf_vectorizer.transform([query_question])
    cosine_sims = cosine_similarity(query_tfidf, search_space_tfidf).flatten()
    k_candidates = min(candidate_list_size, len(search_space))

    if k_candidates < len(cosine_sims):

        candidate_indices = np.argpartition(cosine_sims, -k_candidates)[-k_candidates:]
    else:
        candidate_indices = np.arange(len(cosine_sims))

    candidate_search_space = [search_space[i] for i in candidate_indices]

    if not candidate_search_space:
        return []

    top_k_results = find_top_k_duplicates(
        query_question=query_question,
        model=model,
        tokenizer=tokenizer,
        search_space=candidate_search_space,
        device=device,
        top_k=top_k,
        batch_size=batch_size
    )

    return top_k_results

In [33]:
all_showcase_results_optimized = []

for query in example_queries:
    top_5_results = find_top_k_duplicates_optimized(
        query,
        model,
        tokenizer,
        search_space,
        device,
        tfidf_vectorizer,
        search_space_tfidf,
        top_k=5,
        batch_size=128,
        candidate_list_size=100 #
    )

    all_showcase_results_optimized.append({
        "Query": query,
        "Results": top_5_results
    })

Поиск: 100%|██████████| 1/1 [00:00<00:00, 45.33it/s]


In [34]:
for item in all_showcase_results_optimized:
    print(f"Query: {item['Query']}")
    df = pd.DataFrame(item['Results'], columns=['Logit', 'Found question'])
    df['Logit'] = df['Logit'].round(2)
    print(df.to_string(index=False))

Query: What is the step by step guide to invest in markets in Russia?
    Logit                                                                     Found question
-1.799805                          What is the step by step guide to invest in share market?
-1.940430                                            How can one learn hacking step by step?
-1.969727                                         How do I make my own website step by step?
-2.070312 Which is the step by step process to learn programming from beginning to advanced?
-2.099609                               How do I increase my level of pull ups step by step?
Query: Why is the sky blue?
   Logit                                                                                                 Found question
1.099609                                                                                Why is that the sky is so blue?
0.990234 Why does the sky appear blue? Why is the colour of the sky at the horizon different during sunr

/usr/local/lib/python3.12/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.12/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.12/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.12/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.12/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


As we can see, we got the same candidates when using TF-IDF acceleration, but instead of 1-2 minutes of execution, it took less than 1 second.